# Ciência de Dados em Python — Guia geral de estudo (versão 0.3)

**Objetivo:** aprender os fundamentos desde o início e consultar explicações, comparações e exemplos durante exercícios. O conteúdo segue os cinco temas da disciplina, mas **não depende de questões particulares**. Os PDFs da Estácio orientam a cobertura dos Temas 3 e 4; as explicações incluem complementos conceituais quando necessário.

**Como usar:** leia o tema; pesquise palavras-chave com Ctrl+F; execute primeiro a célula de importações; altere somente os valores indicados nos exemplos. O índice alfabético está ao final.

**Roteiro:** 1. Fundamentos → 2. CNN → 3. Visão computacional → 4. PLN → 5. Reconhecimento de voz → Índice.

## Preparação — importações

**Execute a próxima célula uma vez** ao abrir o notebook. NumPy serve para matrizes; math para funções matemáticas; Counter para contagem de palavras. Se NumPy não estiver instalado, execute em outra célula: `%pip install numpy`. Os exemplos não exigem TensorFlow.

In [ ]:
import numpy as np
import math
from collections import Counter
print('Bibliotecas importadas com sucesso')

## 1. Introdução — fundamentos

**IA** é o campo amplo de sistemas capazes de realizar tarefas associadas à inteligência. **Machine learning** aprende padrões a partir de dados. **Deep learning** é uma família de técnicas de ML que usa redes neurais profundas.

**Supervisionado:** aprende com entradas e respostas corretas (rótulos), por exemplo imagem → gato. **Não supervisionado:** identifica padrões sem rótulos, como agrupamentos. **Por reforço:** aprende por recompensas.

**Treino** é o conjunto usado para atualizar os pesos; **validação** é usado para acompanhar a qualidade e escolher hiperparâmetros, sem atualizar os pesos; **teste** mede desempenho final em dados não vistos. **Overfitting:** aprende detalhes demais do treino e piora em dados novos. Dropout, regularização, data augmentation e early stopping podem ajudar.

**Parâmetros** são aprendidos (pesos, vieses). **Hiperparâmetros** são escolhas de configuração (learning rate, momentum, número de épocas, tamanho do lote).

[Google — fundamentos de machine learning](https://developers.google.com/machine-learning/crash-course) · [Google — overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overview)

## 2. Redes neurais convolucionais (CNNs) — fundamentos explicados

### Pesos sinápticos: o que são?

Um neurônio artificial recebe entradas numéricas, multiplica cada entrada por um **peso sináptico** e soma os resultados com um **viés** (*bias*). O peso indica a influência de uma entrada; o nome é uma analogia com conexões entre neurônios biológicos. Exemplo: entradas 2 e 3, pesos 0,5 e −1, viés 1 → soma = 2×0,5 + 3×(−1) + 1 = −1. A função de ativação transforma essa soma.

**Os pesos são atualizados durante o treinamento? SIM.** O algoritmo calcula gradientes da função de erro e o otimizador modifica pesos e vieses para melhorar previsões. Em geral, na fase de teste/inferência, os pesos ficam fixos. **Não confunda:** pesos são parâmetros aprendidos; learning rate e momentum são hiperparâmetros.

### Treinamento: sequência de operações

1. **Forward pass:** a rede recebe os dados e produz previsões.
2. **Loss (função de erro):** compara previsões aos rótulos corretos.
3. **Backpropagation:** calcula como a loss muda em relação aos pesos (gradientes).
4. **Otimizador:** atualiza os pesos usando esses gradientes.
5. **Validação:** calcula previsões e loss em dados separados, **sem atualização dos pesos**.

**Learning rate (taxa de aprendizado)** controla o tamanho das atualizações; pode ser alterada durante o treinamento por um *scheduler*, ***MAS ATENÇÃO***, tem uma questão que considera que a taxa de aprendizado **NÃO** pode ser alterada durante o treinamento, nessa questão a resposta é "Os pesos sinápticos são atualizados durante o treinamento". Acredito que é considerado errado pois não se altera a taxa de aprendizado durante a iteração, mas é bem confuso isso.

**Momentum** incorpora a direção de atualizações anteriores, suaviza oscilações e **pode acelerar convergência**, mas não garante isso em todos os casos. Portanto, a frase 'momentum acelera a convergência' não é necessariamente falsa; em questão de resposta única, 'pesos são atualizados' descreve mais diretamente o treinamento.

**Fine-tuning** é continuar treinando um modelo pré-treinado, ajustando alguns ou todos os pesos.

### Função de erro (loss): quando é aplicada?

A loss **mede a diferença entre a previsão e a resposta correta**. É calculada no **treino** (para orientar atualização dos pesos) e na **validação** (para avaliar generalização, sem atualizar pesos). Também pode ser medida no teste como avaliação final. Logo, **não é exclusiva da fase de testes**. Exemplos: entropia cruzada para classificação, erro quadrático médio para regressão. **Loss não é ReLU, lambda ou normalização.**

[Google — redes neurais](https://developers.google.com/machine-learning/crash-course/neural-networks) · [PyTorch — otimização](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) · [Keras — losses](https://keras.io/api/losses/)

### Funções de ativação — explicação individual

Uma **função de ativação** transforma o valor produzido por neurônios ou camadas. Em camadas internas, muitas introduzem não linearidade, permitindo aprender padrões complexos. São usadas durante **treinamento e predição**; durante o treino influenciam gradientes, mas **não são o otimizador que atualiza pesos**.

**ReLU:** $f(x)=\\max(0,x)$. Valor negativo vira 0; positivo permanece. Comum em camadas internas de CNNs. Exemplo: [−2, 0, 3] → [0, 0, 3]. Não mede erro.

**Sigmoid:** $f(x)=1/(1+e^{-x})$. Produz valor entre 0 e 1; usada com frequência na saída de classificação binária e multirrótulo, quando cada classe é independente.

**Tanh:** produz valores entre −1 e 1, centrados em zero; comum em componentes de redes recorrentes.

**Softmax:** transforma um **vetor de escores brutos (logits)** em probabilidades não negativas cuja **soma é 1**. Fórmula: $softmax(z_i)=e^{z_i}/\\sum_j e^{z_j}$. Exemplo: logits [2, 1, 0] → aproximadamente [0,665; 0,245; 0,090]. É aplicada principalmente à **saída de classificação multiclasse exclusiva** (uma classe por imagem). Durante o treino, softmax pode fazer parte do cálculo da entropia cruzada; na predição, ajuda a interpretar as saídas. **Softmax não modifica diretamente pesos, não é loss e não é normalização dos dados de entrada.** A maior probabilidade indica a classe escolhida, não certeza absoluta.

**Resumo:** ReLU = zera negativos; sigmoid = valores 0–1 independentes; softmax = distribuição entre classes; loss = mede erro; otimizador = atualiza pesos.

[Google — funções de ativação](https://developers.google.com/machine-learning/crash-course/neural-networks/activation-functions) · [PyTorch — Softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html) · [Vídeo — redes neurais explicadas visualmente (3Blue1Brown)](https://www.youtube.com/watch?v=aircAruvnKk)

### Convolução e estrutura de uma CNN

Uma **CNN** aplica filtros (*kernels*) sobre regiões locais da imagem. **Convolução** combina dois conjuntos de números — entrada e filtro — para produzir um mapa de características (*feature map*). Em cada posição, multiplica elementos correspondentes e soma. Os valores dos filtros são **pesos aprendidos** durante o treinamento.

**Stride** é o passo de deslocamento; **padding** acrescenta bordas; **pooling/aglomeração** reduz a resolução espacial, por exemplo **max pooling** conserva o maior valor de cada região. **ReLU** frequentemente segue uma convolução. Arquitetura típica: imagem → blocos de convolução + ativação + pooling → classificador → previsão. Nem toda convolução precisa ser seguida imediatamente de pooling.

**Dimensão da saída** (sem dilatação): $O=\\lfloor(I+2P-K)/S\\rfloor+1$. I = tamanho da entrada; P = padding de cada lado; K = kernel; S = stride.

[Stanford — convolução e CNNs](https://cs231n.github.io/convolutional-networks/) · [TensorFlow — CNN passo a passo](https://www.tensorflow.org/tutorials/images/cnn)

### Exemplo de código A — calcular valores da convolução

**O que resolve:** dada uma matriz e um filtro, calcula os números produzidos quando o filtro percorre a matriz. Exemplo: na primeira posição, entrada [[1,2],[4,5]] e filtro [[1,0],[0,−1]] → 1×1+2×0+4×0+5×(−1)=−4. **Altere** `matriz` e `kernel` conforme o exercício. `stride` controla o deslocamento; `padding` acrescenta bordas. O padrão é a **correlação cruzada usada em CNNs**; `inverter_kernel=True` faz a convolução matemática com filtro invertido.

In [ ]:

def aplicar_filtro(matriz, kernel, stride=1, padding=0, inverter_kernel=False):
    x = np.asarray(matriz, dtype=float)
    k = np.asarray(kernel, dtype=float)
    if inverter_kernel:  # convolução matemática verdadeira
        k = np.flip(k)
    x = np.pad(x, padding)
    h, w = k.shape
    saida = [[np.sum(x[i:i+h, j:j+w] * k)
              for j in range(0, x.shape[1]-w+1, stride)]
             for i in range(0, x.shape[0]-h+1, stride)]
    return np.array(saida)

matriz = [[1,2,3],[4,5,6],[7,8,9]]  # ALTERE
kernel = [[1,0],[0,-1]]               # ALTERE
print(aplicar_filtro(matriz, kernel, stride=1, padding=0))
print('Convolução com kernel invertido:', aplicar_filtro(matriz, kernel, inverter_kernel=True))

### Exemplo de código B — calcular APENAS o tamanho da saída

**Este código não aplica o filtro nem calcula valores da matriz.** Ele determina quantas posições existirão em um eixo da saída. Exemplo: entrada 32, kernel 3, padding 1, stride 1 → saída 32. **Altere** I, K, P e S; para altura e largura diferentes, faça duas contas.

In [ ]:
I, K, P, S = 32, 3, 1, 1  # ALTERE para cada dimensão
O = math.floor((I + 2*P - K)/S) + 1
print('Tamanho da saída:', O)

### VGGNet e GoogLeNet — diferenças decisivas

**VGGNet (VGG-16/VGG-19):** sua marca é usar **várias convoluções pequenas 3×3 empilhadas**. Elas são organizadas em **blocos**: duas ou três convoluções com ReLU e **depois** uma camada de **max pooling**. Isso se repete, aumentando canais e reduzindo resolução. **VGGNet original usa camadas completamente conectadas? SIM.** Na VGG-16 clássica há **13 camadas convolucionais + 3 totalmente conectadas = 16 camadas com pesos**. A última camada totalmente conectada produz os escores de classificação. Não é correto afirmar que toda convolução é imediatamente seguida por pooling: o pooling vem após grupos de convoluções.

**GoogLeNet (Inception v1):** é caracterizada pelos **módulos Inception** ('módulos de iniciação' em algumas traduções). **Dentro de cada módulo**, a mesma entrada percorre **ramos paralelos** com convoluções de diferentes tamanhos (como 1×1, 3×3 e 5×5) e pooling; as saídas são concatenadas. **Isso NÃO significa que todas as convoluções da rede sejam paralelas**: os módulos se sucedem e também existem camadas sequenciais. A rede original **tem dropout**, mas **dropout não é a característica que a define**. Ela usa *global average pooling* para reduzir parâmetros no classificador.

**Comparação para provas:** VGG = blocos de 3×3 + pooling + camadas fully connected; GoogLeNet = módulos Inception com ramos paralelos **internos**; ResNet = conexões residuais; LeNet = CNN histórica para dígitos; AlexNet = marco do ImageNet 2012.

**Armadilha:** 'há convoluções paralelas nos módulos Inception' é verdadeiro; 'todas as camadas convolucionais da GoogLeNet são paralelas' é falso.

[Artigo VGG original](https://arxiv.org/abs/1409.1556) · [Artigo GoogLeNet original](https://arxiv.org/abs/1409.4842) · [Keras — VGG](https://keras.io/api/applications/vgg/)

## 3. IA aplicada à visão computacional

### 3.1 O que é e onde se aplica

**Visão computacional** permite a um programa interpretar imagens e vídeos: identificar objetos, reconhecer padrões, localizar elementos e compreender cenas. **Processamento de imagens** transforma sinais visuais (redimensionar, filtrar, realçar, detectar bordas); **aprendizado de máquina** aprende relações a partir de exemplos. Os dois podem trabalhar juntos, mas não são sinônimos.

**Aplicações:** análise de exames médicos, monitoramento industrial, veículos autônomos, inspeção de produtos, segurança e análise de estoque no varejo. **Desafios:** iluminação variável, objetos parcialmente ocultos (*oclusão*), ruído, distorção, privacidade e generalização a imagens novas.

### 3.2 Como o computador representa uma imagem

Uma imagem digital pode ser representada por uma **matriz de pixels**. Em tons de cinza, cada posição registra intensidade; em RGB, há três canais (vermelho, verde e azul). **Extração de características** procura padrões como bordas, texturas, formas e cores. Filtros convolucionais extraem padrões locais; camadas posteriores podem combiná-los em características mais abstratas.

**Representação de objetos:** classificação atribui categoria à imagem; detecção identifica **onde** estão objetos (caixas delimitadoras); reconhecimento associa identidade ou categoria; **modelagem 3D** reconstrói geometria tridimensional a partir de imagens 2D. Uma cena também pode ser representada por objetos, posições e relações espaciais.

### 3.3 Segmentação: o que cada tipo entrega

**Segmentação de imagem** separa regiões. **Semântica:** atribui uma classe a **cada pixel**, por exemplo estrada, pedestre ou veículo; duas pessoas distintas recebem a mesma classe. **De instâncias:** além da classe, identifica **cada objeto individualmente** (pessoa A e pessoa B). **Panóptica:** combina regiões semânticas com identidades de objetos. Segmentação é útil em diagnóstico médico, agricultura, robótica e direção autônoma.

**Distinção importante:** classificação = um rótulo para a imagem; detecção = caixas; segmentação = rótulos por pixel; segmentação de instâncias = pixels + indivíduos.

[OpenCV — tutoriais de imagens](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html) · [TensorFlow — segmentação de imagens](https://www.tensorflow.org/tutorials/images/segmentation)

### 3.4 Arquiteturas de visão computacional

**CNN:** aprende filtros locais e forma uma hierarquia de características. **Camada convolucional:** extrai padrões; **ativação:** introduz não linearidade; **pooling:** reduz resolução; **camada totalmente conectada:** combina características para classificação/regressão em arquiteturas clássicas. CNNs podem realizar classificação, detecção e segmentação.

**RNN:** rede recorrente que mantém estado ao processar sequências; embora associada a texto e áudio, pode analisar **sequências de quadros de vídeo** ou eventos visuais no tempo. **GAN:** possui **gerador** (cria imagens sintéticas) e **discriminador** (tenta distinguir imagens reais de geradas); ambos competem durante treinamento. Usos: síntese de imagens, aumento de dados e restauração. **Não confundir:** GAN gera dados; CNN é uma família de arquitetura frequentemente usada para extrair características; RNN modela sequências.

**Computação em nuvem:** permite armazenar dados e executar treinamento/inferência com recursos sob demanda; vantagens incluem **escalabilidade e flexibilidade**. É uma infraestrutura de execução, não uma arquitetura de rede neural. O desenvolvimento pode ocorrer localmente ou em plataformas de nuvem.

### 3.5 Ciclo de desenvolvimento de modelos

1. **Coleta e rotulagem:** imagens e respostas esperadas.
2. **Pré-processamento:** redimensionar, normalizar pixels (por exemplo dividir valores de 0–255 por 255), remover ruídos e selecionar regiões de interesse.
3. **Aumento de dados (*data augmentation*):** rotações, espelhamentos e outras transformações adequadas, normalmente **somente no treino**, para ampliar diversidade sem coletar novas imagens.
4. **Construção:** definir camadas, função de perda e otimizador.
5. **Treinamento supervisionado:** atualizar pesos a partir de imagens rotuladas e da perda.
6. **Validação e ajuste:** medir generalização, ajustar hiperparâmetros e arquitetura.
7. **Teste final e implantação:** avaliar em dados separados e usar em aplicações reais.

**Não confundir:** normalizar pixels é uma transformação de entrada; a função de erro compara previsão e rótulo; o otimizador atualiza pesos. O pré-processamento não se limita à normalização.

[TensorFlow — classificação de imagens](https://www.tensorflow.org/tutorials/images/classification) · [Google — conjuntos de dados](https://developers.google.com/machine-learning/crash-course/datasets)

### 3.6 Avaliação de classificadores — matriz de confusão

A **matriz de confusão** cruza classes verdadeiras com classes previstas e permite analisar erros por tipo. Para classificação binária:

| Real / Previsto | Positivo | Negativo |
|---|---:|---:|
| Positivo | VP (acerto positivo) | FN (positivo perdido) |
| Negativo | FP (falso alarme) | VN (acerto negativo) |

**Acurácia:** $(VP+VN)/(VP+VN+FP+FN)$ — proporção de acertos totais. **Precisão (*precision*):** $VP/(VP+FP)$ — entre previsões positivas, quantas estão certas? **Recall/sensibilidade:** $VP/(VP+FN)$ — entre positivos reais, quantos foram detectados? **Especificidade:** $VN/(VN+FP)$ — entre negativos reais, quantos foram corretamente identificados? **F1:** $2VP/(2VP+FP+FN)$ — combina precisão e recall.

**Exemplo:** em detecção de doenças, um falso negativo é uma pessoa doente classificada como saudável; um falso positivo é uma pessoa saudável classificada como doente. Se classes são desbalanceadas, acurácia alta pode esconder muitos erros relevantes. **Precisão** em português às vezes é usada informalmente como acurácia: observe a fórmula exigida.

**Exemplo de código logo abaixo:** substitua VP, VN, FP e FN; a célula calcula todas as métricas. Para matrizes multiclasse, métricas podem ser calculadas por classe e agregadas por média.

[scikit-learn — matriz de confusão](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) · [scikit-learn — métricas](https://scikit-learn.org/stable/modules/model_evaluation.html)

### 3.7 Python na visão computacional

**Python** oferece estruturas de dados, funções e bibliotecas. **NumPy** representa matrizes e realiza cálculos vetorizados; **Matplotlib** visualiza imagens/gráficos; **OpenCV** (`cv2`) lê e transforma imagens e vídeos; **TensorFlow/Keras** e **PyTorch** criam e treinam redes neurais.

Operações típicas do OpenCV: `cv2.imread` lê imagem; `cv2.cvtColor` converte espaço de cores; `cv2.resize` altera dimensões; `cv2.GaussianBlur` suaviza; `cv2.Canny` detecta bordas. **Não confundir detecção de bordas com detecção de objetos:** bordas identificam mudanças locais de intensidade, objetos exigem interpretação adicional.

**Exemplo conceitual:** carregar uma imagem → converter cores → redimensionar → normalizar pixels → alimentar modelo → obter logits/probabilidades → comparar rótulo previsto ao real.

**Cuidado com dependências:** o código de métricas abaixo usa apenas recursos já importados na preparação. Exemplos com OpenCV/TensorFlow exigiriam instalação e suas importações deveriam ser adicionadas **exclusivamente ao bloco inicial**, não às células posteriores.

[OpenCV — documentação](https://docs.opencv.org/4.x/) · [TensorFlow — tutoriais](https://www.tensorflow.org/tutorials)

### Como utilizar o cálculo abaixo

O código seguinte recebe VP (verdadeiros positivos), VN (verdadeiros negativos), FP (falsos positivos) e FN (falsos negativos). Substitua esses quatro valores e execute: imprime acurácia (acertos totais), precisão (acertos entre previstos positivos), recall (positivos reais identificados), especificidade e F1.

In [ ]:
VP, VN, FP, FN = 40, 50, 10, 5  # ALTERE

def dividir(a,b): return a/b if b else float('nan')
acuracia = dividir(VP+VN, VP+VN+FP+FN)
precisao = dividir(VP, VP+FP)
recall = dividir(VP, VP+FN)
especificidade = dividir(VN, VN+FP)
f1 = dividir(2*VP, 2*VP+FP+FN)
for nome, valor in [('Acurácia',acuracia),('Precisão',precisao),('Recall',recall),('Especificidade',especificidade),('F1',f1)]:
    print(f'{nome}: {valor:.4f} ({valor:.2%})')

## 4. Machine learning aplicado ao processamento de linguagem natural (PLN)

### 4.1 O problema do PLN

**PLN** estuda métodos para computadores analisar, compreender e produzir linguagem humana. Exemplos: classificação de textos, análise de sentimentos (positivo/negativo/neutro), tradução, resumo, respostas a perguntas, reconhecimento de entidades (nomes, datas, locais), chatbots e reconhecimento de fala.

**Texto é sequencial:** a ordem das palavras e o contexto influenciam significado. Antes de treinar modelos, palavras e frases são convertidas em **números**: tokens, índices, vetores de frequência, TF-IDF ou *embeddings* (representações vetoriais aprendidas). **Corpus** é uma coleção de textos usada em análise ou treinamento.

**Tokenização** divide texto em unidades; **lematização** obtém forma de dicionário; **stemming** corta partes das palavras sem necessariamente gerar formas válidas; **stopwords** são termos muito frequentes que podem ser removidos em tarefas apropriadas. **NER** (*named entity recognition*) identifica entidades nomeadas; **POS tagging** classifica funções gramaticais.

### 4.2 Redes neurais para texto

**RNN:** recebe tokens em sequência e mantém um **estado oculto**, que representa memória das entradas anteriores. **Pesos são compartilhados ao longo do tempo**. Seu treinamento pode usar **BPTT** (*backpropagation through time*), que propaga gradientes pelos passos da sequência. Vantagem: ordem temporal explícita; limitação: dependências longas e processamento sequencial. **LSTM/GRU** são variantes que melhoram a retenção de contexto.

**CNN para PLN:** aplica filtros a **janelas locais de tokens/embeddings**, capturando padrões como expressões e combinações próximas; pode ser usada em classificação de texto e sentimento. Ao contrário da imagem 2D, normalmente a convolução percorre a dimensão da sequência. Não é exclusiva de visão computacional.

**Transformer:** arquitetura baseada em **atenção**; permite que cada token considere outros tokens da sequência. Ao contrário de RNNs tradicionais, não exige recorrência passo a passo para codificar sequências, favorecendo **paralelização do treinamento**. Usa mecanismos como autoatenção, representações posicionais, conexões residuais e normalização. Nem todo Transformer é encoder-decoder: há modelos encoder-only e decoder-only.

[Hugging Face — conceitos básicos de Transformers](https://huggingface.co/learn/llm-course/pt/chapter1/4) · [Artigo — Attention Is All You Need](https://arxiv.org/abs/1706.03762)

### 4.3 BERT, GPT, pré-treinamento e fine-tuning

**BERT** (*Bidirectional Encoder Representations from Transformers*) é baseado em **encoders** e usa contexto bidirecional. Aplicações clássicas: classificação, reconhecimento de entidades, análise de sentimentos e extração de respostas. **GPT** (*Generative Pre-trained Transformer*) usa arquitetura predominantemente **decoder autoregressiva**: prevê o próximo token com base nos anteriores e gera texto progressivamente. Aplicações: geração, completar texto, diálogo, resumo, tradução e perguntas e respostas.

**Pré-treinamento:** modelo aprende padrões gerais em grande corpus. Em muitos modelos de linguagem isso é **aprendizado autossupervisionado**, pois alvos são derivados dos próprios textos (frequentemente chamado de 'não supervisionado' em materiais introdutórios). **Fine-tuning/ajuste fino:** treinamento adicional para uma tarefa ou domínio específico, modificando alguns ou todos os parâmetros. Nem toda utilização de modelo pré-treinado exige fine-tuning: também é possível usar modelo congelado ou técnicas de prompting.

**Atenção às distinções:** BERT é tradicionalmente mais associado à compreensão bidirecional; GPT à geração autoregressiva; RNN à recorrência e memória de estados; Transformer à atenção e paralelização. **Modelos** não são o mesmo que **frameworks**.

[Artigo BERT](https://arxiv.org/abs/1810.04805) · [Hugging Face — modelos pré-treinados](https://huggingface.co/learn/llm-course/pt/chapter1/3)

### 4.4 Framework, biblioteca, modelo e API

**Biblioteca:** conjunto de funções/componentes usados pelo programa. **Framework:** estrutura mais abrangente que organiza o desenvolvimento e oferece componentes reutilizáveis, abstração e convenções. Na prática, esses termos às vezes se sobrepõem: o material da disciplina chama ferramentas de PLN de *frameworks*. **Modelo** é o algoritmo/estrutura treinada (por exemplo BERT); **API** é uma interface para acessar funcionalidades.

| Ferramenta | Principal finalidade | Exemplos de uso | O que NÃO é |
|---|---|---|---|
| **spaCy** | pipelines linguísticos eficientes | tokenização, POS, lematização, NER | não é uma arquitetura neural específica |
| **Gensim** | modelagem de tópicos e representações vetoriais | Word2Vec, Doc2Vec, LDA | não é uma biblioteca de interface gráfica |
| **scikit-learn** | ML tradicional e avaliação | TF-IDF, Naive Bayes, regressão logística, métricas, divisão treino/teste | não é exclusivamente para PLN |
| **Transformers (Hugging Face)** | carregar e aplicar modelos pré-treinados | BERT, GPT, tokenizadores, pipelines de tarefas | não é sinônimo da arquitetura Transformer, embora a implemente |
| **TensorFlow/Keras** | construir e treinar redes neurais | modelos de classificação de texto, treinamento e inferência | não é biblioteca específica de tokenização |
| **PyTorch** | treinamento de redes neurais | tensores, autograd, redes personalizadas | não é um modelo de linguagem |

**Componentes reutilizáveis** são vantagem geral dos frameworks; não se restringem a ML. Frameworks aceleram desenvolvimento, mas também podem impor convenções e restrições.

[spaCy — guia](https://spacy.io/usage/spacy-101) · [Gensim — documentação](https://radimrehurek.com/gensim/) · [scikit-learn — classificação de textos](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html) · [Transformers — documentação](https://huggingface.co/docs/transformers/index)

### 4.5 Fluxos práticos de PLN em Python

**Análise linguística com spaCy:** carregar modelo de idioma → processar texto → percorrer tokens → consultar texto, classe gramatical, lema e entidades. Modelos de português precisam estar instalados separadamente; o pacote sozinho não garante um pipeline linguístico completo.

**Tópicos e vetores com Gensim:** preparar frases tokenizadas → treinar Word2Vec para representar palavras em vetores → consultar palavras semelhantes; alternativamente usar LDA para descobrir temas em uma coleção de documentos. Word2Vec aprende representações, **não é um classificador de sentimentos pronto**.

**Classificação tradicional com scikit-learn:** documentos rotulados → separar treino/teste → converter textos com CountVectorizer ou TfidfVectorizer → treinar classificador → prever → avaliar precisão/recall/F1. Ajustar vetorizador apenas no treino evita vazamento de informação.

**Modelos Transformers pré-treinados:** escolher modelo adequado e tokenizador → transformar texto em tokens → inferência com pipeline (por exemplo análise de sentimentos) → interpretar rótulos e escores. Os resultados dependem de idioma, domínio, dados e modelo; **nem todo modelo de sentimentos em inglês funciona bem em português**.

**Exemplo prático sem bibliotecas adicionais:** a célula abaixo ilustra tokenização simples, contagem de palavras e frequência. Ela **não substitui tokenizadores linguísticos** (não trata pontuação/subpalavras corretamente), mas mostra por que texto precisa de representação numérica.

[Hugging Face — pipelines](https://huggingface.co/docs/transformers/main_classes/pipelines) · [spaCy — modelos de idiomas](https://spacy.io/models/pt)

In [ ]:
texto = 'Python ajuda a analisar texto e Python ajuda a aprender'
tokens = texto.lower().split()  # ALTERE o texto acima
frequencias = Counter(tokens)
vocabulario = {palavra: i for i, palavra in enumerate(sorted(frequencias))}
print('Tokens:', tokens)
print('Frequências:', frequencias)
print('Índices do vocabulário:', vocabulario)

## 5. IA aplicada ao reconhecimento de voz

### Pipeline clássico de reconhecimento automático de fala (ASR)

1. **Captura/digitalização:** microfone → sinal analógico → amostragem temporal e quantização de amplitude → amostras digitais.
2. **Pré-processamento:** redução de ruído, normalização, recorte e extração de características (ex.: MFCC).
3. **Modelo acústico:** associa características sonoras a fonemas/unidades acústicas.
4. **Modelo léxico / dicionário:** mapeia palavras às sequências de fonemas/pronúncias.
5. **Modelo de linguagem:** atribui probabilidades a sequências de palavras (ex.: n-gramas).
6. **Decodificador:** combina evidências e escolhe transcrição provável.

**Atenção:** modelos end-to-end podem integrar etapas em uma única rede; a divisão acima descreve sistemas clássicos.

- **Grafema:** unidade de escrita (letra/grupo de letras). **Fonema:** unidade distintiva de som. Relação não é sempre 1:1.
- **speech_recognition:** nome do pacote Python usado para interface de reconhecimento de fala (`SpeechRecognition`); **NumPy:** arrays; **Matplotlib:** gráficos; **Tkinter:** interface gráfica. `voice_recognition` não é o nome padrão desse pacote.
- **Lematização:** forma de dicionário das palavras, não é o mesmo que reconhecimento acústico.
- **Unigrama** (n=1): P(palavra); **bigrama** (n=2): P(palavra atual | anterior); **trigrama** (n=3): P(atual | duas anteriores).
- **Perplexidade:** mede quão surpreendente é a sequência para o modelo; **menor é melhor** quando a avaliação é comparável (mesma tokenização e dados).
- **Sinal digital:** frequência de amostragem em Hz = amostras por segundo; resolução em bits determina níveis de quantização (idealmente $2^b$). **Erro de quantização:** diferença entre valor original e representado.
- **Pares ordenados (tempo, amplitude):** pontos de uma forma de onda amostrada. Se questão pedir erro total, identificar se é soma de erros absolutos, quadráticos ou com sinal.

### N-gramas: identificar e calcular probabilidades

Estimativa sem suavização de um bigrama: $P(w_i\mid w_{i-1})=C(w_{i-1},w_i)/C(w_{i-1})$.

Para trigrama: $P(w_i\mid w_{i-2},w_{i-1})=C(w_{i-2},w_{i-1},w_i)/C(w_{i-2},w_{i-1})$.

**Atenção:** contagens devem corresponder ao mesmo contexto de previsão; em aplicações reais são comuns tokens de início/fim e suavização.

In [ ]:
frases = ['eu gosto de python', 'eu gosto de dados', 'eu estudo python']  # ALTERE
n = 2  # 1=unigramas, 2=bigramas, 3=trigramas
contagens = Counter(tuple(tokens[i:i+n]) for frase in frases
                    for tokens in [frase.lower().split()]
                    for i in range(len(tokens)-n+1))
print(contagens)
# Exemplo: P('gosto' | 'eu') = contagem ('eu','gosto') / contagem contextos 'eu'
contexto = ('eu',)
alvo = 'gosto'
numerador = contagens[contexto+(alvo,)]
denominador = sum(v for seq,v in contagens.items() if seq[:-1] == contexto)
print('Probabilidade:', numerador/denominador if denominador else 'contexto ausente')

### Perplexidade

Para probabilidades condicionais $p_1,\ldots,p_N$ dos tokens efetivamente observados:

$$PP=\exp\left(-\frac{1}{N}\sum_{i=1}^N\ln p_i\right)$$

Equivale a $\left(\prod_i p_i\right)^{-1/N}$. Probabilidade zero sem suavização implica perplexidade infinita.

### Como utilizar o cálculo abaixo

O próximo código calcula perplexidade com probabilidades condicionais de palavras observadas. Altere a lista probabilidades usando valores decimais entre 0 e 1 (0,5 = 50%). Menor perplexidade indica menor surpresa, quando a comparação é válida.

In [ ]:
probabilidades = [0.5, 0.25, 0.5]  # ALTERE: probabilidade de cada palavra observada
if any(p < 0 or p > 1 for p in probabilidades) or not probabilidades:
    raise ValueError('Informe probabilidades em (0,1]')
pp = math.inf if 0 in probabilidades else math.exp(-sum(math.log(p) for p in probabilidades)/len(probabilidades))
print('Perplexidade:', pp)

### Matriz de estados / cadeias de Markov

Se **linhas = estado atual** e **colunas = próximo estado**, a entrada $P[i,j]$ é a probabilidade de ir de i para j em um passo. Linhas somam 1.

Distribuição após $n$ passos: $\pi_n=\pi_0P^n$ (vetor linha). Em questões com orientação oposta (colunas = atual), a interpretação muda.

### Como utilizar o cálculo abaixo

O próximo código usa matriz de transição. Linhas representam estado atual, colunas o próximo; cada linha soma 1. Altere P, estado_atual, proximo_estado e passos. Índices começam em 0. Para um passo lê P[i,j]; para vários, calcula P elevado ao número de passos.

In [ ]:
P = np.array([[0.8, 0.2], [0.3, 0.7]])  # ALTERE
estado_atual = 0  # índice 0 = estado 1
proximo_estado = 1  # índice 1 = estado 2
passos = 1
assert np.allclose(P.sum(axis=1), 1), 'Cada linha deve somar 1'
print('Transição em um passo:', P[estado_atual, proximo_estado])
print(f'Transição em {passos} passo(s):', np.linalg.matrix_power(P, passos)[estado_atual, proximo_estado])

### Erros entre sinal original e digitalizado

Se a questão apresentar pares (valor real, valor aproximado), verifique qual medida solicita: erro absoluto total $\sum|x_i-\hat x_i|$; erro quadrático total $\sum(x_i-\hat x_i)^2$; erro médio quadrático $MSE=\frac1N\sum(x_i-\hat x_i)^2$. **Não confundir** com taxa de erro de palavras (WER) no ASR.

### Como utilizar o cálculo abaixo

O próximo código compara valores originais e digitalizados. Altere os dois vetores (mesmo tamanho). Imprime soma de erros com sinal, erros absolutos, quadráticos e MSE. Verifique qual definição de 'erro total' a questão pede.

In [ ]:
original = np.array([1.0, 2.0, 3.0])  # ALTERE
aproximado = np.array([1.1, 1.8, 3.2])  # ALTERE
erros = original - aproximado
print('Erro com sinal (soma):', erros.sum())
print('Erro absoluto total:', np.abs(erros).sum())
print('Erro quadrático total:', np.square(erros).sum())
print('Erro médio quadrático (MSE):', np.square(erros).mean())

## Resumo geral para consulta

**Aprendizado:** supervisionado = dados rotulados; não supervisionado = padrões sem rótulos; parâmetros = pesos aprendidos; hiperparâmetros = escolhas do treinamento.

**CNN:** convolução extrai características; pooling reduz resolução; ReLU ativa; loss mede erro; otimizador atualiza pesos. VGGNet = blocos de convoluções 3×3 e pooling, com topo totalmente conectado clássico; GoogLeNet = módulos Inception com ramos paralelos **internos**.

**Visão computacional:** classificação = rótulo; detecção = localização; segmentação semântica = classes por pixel; instâncias = indivíduos por pixel; GAN = gerador + discriminador; RNN = sequências; nuvem = infraestrutura escalável.

**PLN:** RNN = memória sequencial; CNN textual = padrões locais; Transformer = atenção; BERT = contexto bidirecional; GPT = geração autoregressiva. spaCy = análise linguística; Gensim = tópicos/vetores; scikit-learn = ML clássico; Hugging Face Transformers = modelos pré-treinados.

**Voz:** modelo acústico = som; léxico = pronúncia; linguagem = sequências de palavras; n-grama = sequência de n tokens; perplexidade = surpresa preditiva; matriz de estados = probabilidades de transição.

## Índice alfabético de conceitos

Use **Ctrl+F** pelo termo ou consulte a seção indicada. As entradas apontam para explicações no corpo do material.

| Conceito | Seção | Ideia central |
|---|---|---|
| Acurácia, precisão, recall, F1 | 3.6 | métricas da matriz de confusão |
| Aprendizado supervisionado | 1; 2; 3.5 | rótulos orientam aprendizado |
| Backpropagation | 2 | cálculo de gradientes |
| BERT / GPT | 4.3 | compreensão / geração |
| CNN / convolução / pooling | 2; 3.4 | filtros / redução espacial |
| Dropout / overfitting | 1; 2 | regularização / generalização |
| Fine-tuning | 2; 4.3 | adaptar modelo pré-treinado |
| Função de erro / loss | 2 | comparação previsão-alvo |
| GAN | 3.4 | gerador e discriminador |
| GoogLeNet / Inception | 2 | módulos com ramos paralelos internos |
| Learning rate / momentum | 2 | passo e histórico de atualização |
| Matriz de confusão | 3.6 | classificação real versus prevista |
| N-gramas / perplexidade | 5 | sequência e probabilidade |
| OpenCV / NumPy | 3.7 | processamento de imagens e matrizes |
| Pesos sinápticos | 2 | parâmetros aprendidos |
| ReLU / sigmoid / softmax | 2 | funções de ativação |
| RNN / LSTM / BPTT | 4.2 | sequência, memória e treino temporal |
| Segmentação semântica / instâncias | 3.3 | rótulos de pixels e objetos |
| spaCy / Gensim / scikit-learn | 4.4–4.5 | ferramentas de PLN |
| Transformer / atenção | 4.2 | dependências entre tokens |
| Transição de estados | 5 | probabilidade de mudança |
| VGGNet / fully connected | 2 | blocos 3×3 e classificador |

### Materiais de referência

**Base dos Temas 3 e 4:** PDFs da Estácio *IA aplicada à visão computacional* (54 páginas) e *Machine Learning Aplicado para PLN* (46 páginas), disponibilizados com este estudo. As seções preservam a cobertura de fundamentos, arquiteturas, aplicações e uso de Python; explicações complementares são síntese didática, não transcrição.

**Documentação externa:** [Google ML Crash Course](https://developers.google.com/machine-learning/crash-course), [Stanford CS231n](https://cs231n.github.io/convolutional-networks/), [OpenCV](https://docs.opencv.org/4.x/), [spaCy](https://spacy.io/usage/spacy-101), [Hugging Face](https://huggingface.co/learn/llm-course/pt/chapter1/1).

**Versão 0.3:** estrutura generalizada, remoção de referências a exercícios individuais, importações centralizadas, Temas 3 e 4 expandidos a partir do conteúdo da disciplina.